<a href="https://colab.research.google.com/github/gauravd12345/optiML/blob/main/softmax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [143]:
!nvidia-smi
!nvcc --version

Thu Nov  5 06:02:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [144]:
import numpy as np
import timeit

N = 10
rng = np.random.default_rng()


logits = rng.random(N)
# print(logits)

In [145]:
def softmax_cpu(logits):
    shifted_logits = logits - np.max(logits)
    exp_logits = np.exp(shifted_logits)
    probabilities = exp_logits / np.sum(exp_logits)

    return probabilities


In [146]:
probabilities = softmax_cpu(logits)
# print(probabilities)
# print(f"Sum of proabilities: {np.sum(probabilities)}")

In [147]:
for n in [128, 512, 1024, 4096, 16384, 65536, 100000]:
    logits = np.random.randn(n).astype(np.float32)

    times = timeit.repeat(
        stmt="softmax_cpu(logits)",
        globals={"softmax_cpu": softmax_cpu, "logits": logits},
        repeat=10,
        number=100
    )

    median_ms = np.median(times) * 1000 / 100
    print(f"{n:6d} elements: {median_ms:.4f} ms")

   128 elements: 0.0106 ms
   512 elements: 0.0112 ms
  1024 elements: 0.0126 ms
  4096 elements: 0.0175 ms
 16384 elements: 0.0346 ms
 65536 elements: 0.1493 ms
100000 elements: 0.2030 ms


In [148]:
%%writefile softmax_gpu.cu

#include <stdio.h>
#include <math.h>
#include <stdlib.h>

__global__ void softmax_gpu(double norm, double *logits, double *probabilities, int N){

  int i = blockIdx.x * blockDim.x + threadIdx.x;

  // calculating probabilities
  if(i < N){
    probabilities[i] = exp(logits[i]) / norm;
  }

}


int main(){

  srand(1234);

  int N = 100000;
  int threadsPerBlock = 256;
  int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

  /* initializing logits */
  double logits[N];
  double probabilities[N];
  for (int i = 0; i < N; i++) {
      double u = (double) rand() / RAND_MAX;  // [0, 1]
      logits[i] = 2.0 * u - 1.0;              // [-1, 1]
  }

  /* calculating normalization constant */
  double norm = 0;
  for(int i = 0; i < N; i++){
    norm += exp(logits[i]);
  }

  cudaEvent_t start, stop;
  cudaEventCreate(&start);
  cudaEventCreate(&stop);

  /* allocating memory on GPU */
  double *ld, *pd;
  int size = N * sizeof(double);
  cudaMalloc((void**)&ld, size);
  cudaMalloc((void**)&pd, size);

  // copying data over to the GPU
  cudaMemcpy(ld, logits, size, cudaMemcpyHostToDevice);

  // launching kernel
  int repeats = 100;

  cudaEventRecord(start);

  for (int i = 0; i < repeats; i++) {
      softmax_gpu<<<blocksPerGrid, threadsPerBlock>>>(norm, ld, pd, N);
  }
  cudaEventRecord(stop);
  cudaEventSynchronize(stop);

  // copying probabilities back from GPU
  cudaMemcpy(probabilities, pd, size, cudaMemcpyDeviceToHost);

  cudaFree(ld);
  cudaFree(pd);

  // veriyfying sum
  double sum = 0;
  for(int i = 0; i < N; i++){
    sum += probabilities[i];

  }

  printf("%.2f\n", sum);

  // measuring runtime
  float totalMilliseconds = 0;
  cudaEventElapsedTime(&totalMilliseconds, start, stop);

  printf("Average kernel time: %f ms\n", totalMilliseconds / repeats);
  cudaEventDestroy(start);
  cudaEventDestroy(stop);


}

Overwriting softmax_gpu.cu


In [149]:
%%shell

nvcc softmax_gpu.cu -o softmax_gpu
./softmax_gpu

1.00
Average kernel time: 0.038647 ms
